# ERA5 ARCO Zarr

Open an authenticated ERA5 ARCO Zarr archive and inspect its final time step.

Run the cells in order. Credentials are read from local configuration; this notebook contains no saved credentials.

In [ ]:
import xarray as xr
from ecmwf.datastores.config import get_config


## 1. Read the authentication key


In [ ]:
# Check ECMWF_DATASTORES_KEY first, then ~/.ecmwfdatastoresrc.
# Leave blank to use credentials in ~/.ecmwfdatastoresrc.
# Never commit a notebook containing an API key.
api_key = ""
api_key = api_key or get_config("key")

if not api_key:
    raise RuntimeError(
        "No CDS API key was found. Follow the CDS credential guide, then run "
        "this notebook again."
    )

storage_options = {"headers": {"Authorization": f"Bearer {api_key}"}}


## 2. Choose an archive


In [ ]:
# Geo-chunked is best for long time series at a point or small area.
geochunked_url = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-geo-002/arco/"
    "reanalysis_era5_single_levels/sfc/geoChunked.zarr"
)

# Time-chunked is best for maps or large areas over a short period.
timechunked_url = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-time-002/arco/"
    "reanalysis_era5_single_levels/sfc/timeChunked.zarr"
)

# This notebook selects one time step, so use the time-chunked archive.
selected_url = timechunked_url


## 3. Open the remote Zarr archive


In [ ]:
# consolidated=True reduces metadata requests. Opening remains lazy.
ds = xr.open_zarr(
    selected_url,
    consolidated=True,
    storage_options=storage_options,
)
ds


## 4. Select the final time step


In [ ]:
# isel selects by position; -1 means the final time entry.
latest = ds.isel(time=-1)
print("FINAL TIME COORDINATE")
print(latest["time"].values)


## 5. Inspect the selected data


In [ ]:
print("DATASET STRUCTURE")
print(latest)

full_size = ds.nbytes
last_size = latest.nbytes
print(f"\nFull dataset:   {full_size / 10**12:.2f} TB")
print(f"Last time step: {last_size / 10**6:.2f} MB")

# To retrieve actual values, select a variable first, for example:
# temperature = latest["2m_temperature"].load()


## 6. Close the archive


In [ ]:
ds.close()
